## 바이오 빅데이터 분석 (5B)
### ANN 회귀 실습: Morgan fingerprint, ChemBERTa embedding, Tiny SMILES-BERT embedding 기반 CCLE-CTRPv2 AUC 예측 비교

이번 실습에서는 4B에서 만든 `expression + mutation + compound representation -> AUC` 문제를 확장합니다. 기존 ML 모델 비교는 제외하고, ANN 안에서 compound representation만 바꾸어 비교합니다.

```text
Model A: gene expression + mutation + Morgan fingerprint -> ANN -> AUC
Model B: gene expression + mutation + ChemBERTa embedding -> ANN -> AUC
Model C: gene expression + mutation + 5A Tiny SMILES-BERT embedding -> ANN -> AUC
```

#### 학습 목표

- CTRPv2 compound SMILES에서 Morgan fingerprint, ChemBERTa embedding, Tiny SMILES-BERT embedding을 각각 계산한다.
- 5A에서 QM9로 학습한 Tiny SMILES-BERT의 weight, vocabulary, config를 불러와 새로운 CTRPv2 compound embedding을 만든다.
- 같은 cell line feature와 같은 train/validation/test split에서 ANN 성능을 비교한다.
- pretrained Transformer embedding과 작은 교육용 Transformer embedding이 downstream task에서 어떻게 달라지는지 토의한다.


### 0. 실습 환경

필요 패키지는 `rdkit`, `tensorflow`, `torch`, `transformers`입니다.

ChemBERTa는 Hugging Face model을 사용하므로 처음 실행할 때 인터넷 다운로드가 필요할 수 있습니다. 이미 로컬 캐시에 있으면 인터넷 없이도 실행될 수 있습니다.

Tiny SMILES-BERT embedding을 계산하려면 5A 노트북 마지막의 모델 저장 셀을 먼저 실행해 `outputs/tiny_smiles_bert_model` 폴더에 weight, vocabulary, config 파일을 만들어두어야 합니다.


In [ ]:
# 필요할 때만 실행하세요.
# !conda install -c conda-forge rdkit tensorflow pytorch -y
# !pip install transformers


In [ ]:
from pathlib import Path
import importlib.util
import json
import re
import random
import warnings

import numpy as np
import pandas as pd
pd.set_option("future.infer_string", False)

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

from sklearn import metrics as met
from sklearn import model_selection as mod_sel
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


In [ ]:
if importlib.util.find_spec("rdkit") is None:
    raise ImportError("rdkit이 필요합니다. 설치 셀을 참고하세요.")
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog("rdApp.warning")
RDLogger.DisableLog("rdApp.error")

TF_AVAILABLE = importlib.util.find_spec("tensorflow") is not None
if TF_AVAILABLE:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    tf.random.set_seed(RANDOM_STATE)
    print("TensorFlow version:", tf.__version__)
else:
    print("TensorFlow is not installed. ANN section requires tensorflow.")

TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None
TRANSFORMERS_AVAILABLE = importlib.util.find_spec("transformers") is not None
if TORCH_AVAILABLE:
    import torch
    import torch.nn as nn
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Torch device:", device)
else:
    device = None
    print("torch is not installed. ChemBERTa and Tiny SMILES-BERT embeddings require torch.")


## 1. 데이터 불러오기와 문제 정의

예측 문제는 4B와 동일합니다.

```text
cell line feature + compound feature -> AUC(cell line, compound)
```

5B에서는 compound feature를 Morgan fingerprint와 ChemBERTa embedding 두 가지로 만들어 비교합니다.


In [ ]:
def find_data_dir():
    """Return the local folder that contains CCLE-CTRPv2 CSV files."""
    candidates = [Path("data_CCLE_CTRPv2"), Path("Work_BioBigdata_KOTHEA/data_CCLE_CTRPv2")]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def load_ccle_ctrpv2(base_dir=None):
    """Load CCLE-CTRPv2 tables from local CSV files or mlbi-lab fallback."""
    base = Path(base_dir) if base_dir is not None else find_data_dir()
    files = {
        "gene_expression": base / "ccle_ctrpv2_gene_exp.csv",
        "cellline_info": base / "ccle_ctrpv2_cellline_info.csv",
        "auc": base / "ccle_ctrpv2_auc.csv",
        "ec50": base / "ccle_ctrpv2_ec50.csv",
        "drug_info": base / "ccle_ctrpv2_drug_info.csv",
        "mutation_info": base / "ccle_ctrpv2_mutation_info.csv",
    }
    if all(path.exists() for path in files.values()):
        print("Loading local CSV files from", base.resolve())
        return {key: pd.read_csv(path, index_col=0) for key, path in files.items()}
    try:
        from mlbi.data import load_data
        print("Local CSV files were not found. Loading data with mlbi-lab fallback.")
        return load_data("ccle-ctrpv2")
    except Exception as exc:
        raise FileNotFoundError("Check data_CCLE_CTRPv2/ or install mlbi-lab.") from exc


data = load_ccle_ctrpv2()
df_gexp = data["gene_expression"]
df_cl_info = data["cellline_info"]
df_auc = data["auc"]
df_drug_info = data["drug_info"]
df_mutation_info = data["mutation_info"]

print("gene expression:", df_gexp.shape)
print("AUC            :", df_auc.shape)
print("drug info      :", df_drug_info.shape)
print("mutation info  :", df_mutation_info.shape)


## 2. 데이터 전처리

전처리 흐름은 4B와 거의 같습니다.

1. 사용할 compound와 cell line을 고릅니다.
2. compound protein target gene으로 mutation matrix를 만듭니다.
3. expression + mutation cell line feature를 만듭니다.
4. compound SMILES에서 Morgan fingerprint와 ChemBERTa embedding을 각각 만듭니다.
5. AUC matrix를 long-form table로 바꾸고 feature를 병합합니다.


### 2-1. 사용할 compound와 cell line 선택


In [ ]:
common_compounds = df_auc.columns.intersection(df_drug_info.index)
smiles_mask = df_drug_info.loc[common_compounds, "cpd_smiles"].notna()
usable_compounds = df_drug_info.loc[common_compounds].loc[smiles_mask].index.tolist()

# ChemBERTa embedding 계산 시간을 고려해 기본값은 AUC sample 수가 많은 compound 80개만 사용합니다.
MAX_COMPOUNDS = 80
if MAX_COMPOUNDS is not None:
    compound_counts = df_auc[usable_compounds].notna().sum().sort_values(ascending=False)
    usable_compounds = compound_counts.head(MAX_COMPOUNDS).index.tolist()

usable_cell_lines = df_gexp.index.intersection(df_auc.index).tolist()
print("Usable cell lines:", len(usable_cell_lines))
print("Usable compounds :", len(usable_compounds))
print("Observed AUC pairs:", int(df_auc.loc[usable_cell_lines, usable_compounds].notna().sum().sum()))


### 2-2. target-gene mutation matrix 만들기


In [ ]:
def parse_target_gene_string(value):
    """Parse a semicolon-separated target-gene string into a clean gene list."""
    if pd.isna(value):
        return []
    return [gene.strip() for gene in str(value).split(";") if gene.strip()]


def as_boolean(series):
    """Convert mixed boolean-like values such as True, 1, yes into bool values."""
    return series.fillna(False).astype(str).str.lower().isin(["true", "1", "yes"])


def make_target_mutation_matrix(mutation_info, target_genes, sample_index):
    """Build a cell line x target-gene mutation feature matrix."""
    target_genes = [gene for gene in target_genes if gene in set(mutation_info["Hugo_Symbol"])]
    feature_columns = []
    for gene in target_genes:
        feature_columns.extend([f"mut_{gene}_any", f"mut_{gene}_damaging", f"mut_{gene}_hotspot"])

    mutation_matrix = pd.DataFrame(0, index=sample_index, columns=feature_columns, dtype=np.int8)
    if not target_genes:
        return mutation_matrix

    mut = mutation_info.loc[
        mutation_info["Hugo_Symbol"].isin(target_genes)
        & mutation_info["ccl_name"].isin(sample_index)
    ].copy()
    if mut.empty:
        return mutation_matrix

    mut["is_damaging_feature"] = as_boolean(mut["isDeleterious"])
    mut["is_hotspot_feature"] = as_boolean(mut["isTCGAhotspot"]) | as_boolean(mut["isCOSMIChotspot"])

    for gene in target_genes:
        gene_mut = mut[mut["Hugo_Symbol"] == gene]
        if gene_mut.empty:
            continue
        any_cells = gene_mut["ccl_name"].unique()
        damaging_cells = gene_mut.loc[gene_mut["is_damaging_feature"], "ccl_name"].unique()
        hotspot_cells = gene_mut.loc[gene_mut["is_hotspot_feature"], "ccl_name"].unique()
        mutation_matrix.loc[mutation_matrix.index.isin(any_cells), f"mut_{gene}_any"] = 1
        mutation_matrix.loc[mutation_matrix.index.isin(damaging_cells), f"mut_{gene}_damaging"] = 1
        mutation_matrix.loc[mutation_matrix.index.isin(hotspot_cells), f"mut_{gene}_hotspot"] = 1
    return mutation_matrix


compound_target_genes = {
    compound: parse_target_gene_string(df_drug_info.loc[compound, "gene_symbol_of_protein_target"])
    for compound in usable_compounds
}
all_target_genes = sorted({gene for genes in compound_target_genes.values() for gene in genes})
target_genes_in_expression = [gene for gene in all_target_genes if gene in df_gexp.columns]
target_genes_in_mutation = [gene for gene in all_target_genes if gene in set(df_mutation_info["Hugo_Symbol"])]

df_mut_features = make_target_mutation_matrix(df_mutation_info, target_genes_in_mutation, usable_cell_lines)
print("Mutation feature matrix:", df_mut_features.shape)


### 2-3. expression + mutation cell line feature 만들기


In [ ]:
df_log_gexp = np.log10(df_gexp.loc[usable_cell_lines] + 1)

TOP_N_GENES = 500
if TOP_N_GENES is None or TOP_N_GENES >= df_log_gexp.shape[1]:
    selected_expression_genes = df_log_gexp.columns.tolist()
else:
    selected_expression_genes = df_log_gexp.var(axis=0).sort_values(ascending=False).head(TOP_N_GENES).index.tolist()
selected_expression_genes = sorted(set(selected_expression_genes).union(target_genes_in_expression))

df_expr_features = df_log_gexp.loc[:, selected_expression_genes].copy()
df_expr_features.columns = [f"expr_{gene}" for gene in df_expr_features.columns]

cellline_features = pd.concat([df_expr_features, df_mut_features], axis=1).copy()
expr_cols = df_expr_features.columns.tolist()
mut_cols = df_mut_features.columns.tolist()
cell_cols = cellline_features.columns.tolist()

print("Expression features:", len(expr_cols))
print("Mutation features  :", len(mut_cols))
print("Cell line features :", cellline_features.shape)


### 2-4. Morgan fingerprint 만들기


In [ ]:
def morgan_fingerprint_from_smiles(smiles, radius=2, n_bits=512):
    """Convert one SMILES string into a Morgan fingerprint bit vector."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


MORGAN_RADIUS = 2
MORGAN_BITS = 512
fp_rows = []
valid_fp_compounds = []
for compound in usable_compounds:
    arr = morgan_fingerprint_from_smiles(df_drug_info.loc[compound, "cpd_smiles"], MORGAN_RADIUS, MORGAN_BITS)
    if arr is not None:
        valid_fp_compounds.append(compound)
        fp_rows.append(arr)

fp_cols = [f"fp_{i}" for i in range(MORGAN_BITS)]
df_fp_features = pd.DataFrame(np.vstack(fp_rows), index=valid_fp_compounds, columns=fp_cols)
print("Morgan fingerprint matrix:", df_fp_features.shape)


### 2-5. ChemBERTa embedding 만들기

CTRPv2 compound SMILES를 pretrained ChemBERTa encoder에 넣고 `[CLS]` embedding을 compound representation으로 사용합니다. 처음 실행 시 model 다운로드가 필요할 수 있습니다.


In [ ]:
def compute_chemberta_embeddings(smiles_series, model_name="DeepChem/ChemBERTa-77M-MLM", batch_size=64, max_length=128):
    """Compute pretrained ChemBERTa CLS embeddings for a Series of SMILES."""
    if not TORCH_AVAILABLE or not TRANSFORMERS_AVAILABLE:
        raise ImportError("ChemBERTa embedding requires torch and transformers.")
    from transformers import AutoModel, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    smiles_list = smiles_series.astype(str).tolist()
    embeddings = []
    with torch.no_grad():
        for start in range(0, len(smiles_list), batch_size):
            batch_smiles = smiles_list[start : start + batch_size]
            encoded = tokenizer(
                batch_smiles,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            output = model(**encoded)
            cls_embedding = output.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embedding)
    return np.vstack(embeddings).astype(np.float32)


CHEMBERTA_MODEL_NAME = "DeepChem/ChemBERTa-77M-MLM"
chemberta_compounds = valid_fp_compounds
chemberta_smiles = df_drug_info.loc[chemberta_compounds, "cpd_smiles"]

X_chemberta = compute_chemberta_embeddings(
    chemberta_smiles,
    model_name=CHEMBERTA_MODEL_NAME,
    batch_size=64,
    max_length=128,
)
chemberta_cols = [f"chemberta_{i}" for i in range(X_chemberta.shape[1])]
df_chemberta_features = pd.DataFrame(X_chemberta, index=chemberta_compounds, columns=chemberta_cols)
print("ChemBERTa embedding matrix:", df_chemberta_features.shape)


### 2-6. Tiny SMILES-BERT embedding 만들기

5A에서 QM9 데이터로 학습한 작은 Transformer encoder를 불러와 CTRPv2 compound SMILES의 `[CLS]` embedding을 새로 계산합니다.

여기서 사용하는 것은 5A에서 이미 계산된 QM9 embedding matrix가 아니라, 후속 데이터에 다시 적용할 수 있는 `weight + vocabulary + config`입니다. CTRPv2 compound는 QM9보다 구조가 복잡하므로, 5A vocabulary에 없는 token은 `[UNK]`로 처리됩니다.


In [ ]:
SMILES_TOKEN_PATTERN = re.compile(
    r"(\[[^\]]+\]|Br|Cl|Si|Se|Li|Na|Ca|Mg|Al|[B-IK-NO-Z][a-z]?|[bcnops]|"
    r"\%\d{2}|\d|\(|\)|\.|=|#|-|\+|\\|/|:|~|@|\?)"
)


def tokenize_smiles_for_tiny_model(smiles):
    """Tokenize a SMILES string using the same rule-based tokenizer as 5A."""
    tokens = SMILES_TOKEN_PATTERN.findall(str(smiles))
    if "".join(tokens) != str(smiles):
        raise ValueError(f"토큰화하지 못한 SMILES가 있습니다: {smiles} -> {tokens}")
    return tokens


class TinySmilesBERT(nn.Module):
    """Small Transformer encoder with the same architecture as the 5A Tiny SMILES-BERT."""
    def __init__(self, vocab_size, max_len, pad_id, dim=96, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, dim, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_len, dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=n_heads,
            dim_feedforward=dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(dim)
        self.mlm_head = nn.Linear(dim, vocab_size)

    def forward(self, input_ids, attention_mask):
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, seq_len)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        key_padding_mask = ~attention_mask
        hidden = self.encoder(x, src_key_padding_mask=key_padding_mask)
        hidden = self.norm(hidden)
        logits = self.mlm_head(hidden)
        return logits, hidden


def find_tiny_model_dir():
    """Find the Tiny SMILES-BERT files saved by the 5A notebook."""
    candidates = [
        Path("Work_BioBigdata_KOTHEA/outputs/tiny_smiles_bert_model"),
        Path("outputs/tiny_smiles_bert_model"),
        Path("embedding_outputs/tiny_smiles_bert_model"),
    ]
    for path in candidates:
        if (path / "tiny_smiles_bert_state_dict.pt").exists():
            return path
    raise FileNotFoundError(
        "5A에서 Tiny SMILES-BERT 모델 저장 셀을 먼저 실행해 주세요. "
        "필요 파일: tiny_smiles_bert_state_dict.pt, tiny_smiles_bert_vocab.json, tiny_smiles_bert_config.json"
    )


def encode_tokens_for_tiny_model(tokens, stoi, max_len, pad_id, cls_id, sep_id, unk_id):
    """Convert SMILES tokens into padded token ids and an attention mask."""
    ids = [cls_id] + [stoi.get(tok, unk_id) for tok in tokens[: max_len - 2]] + [sep_id]
    attention_mask = [1] * len(ids)
    pad_len = max_len - len(ids)
    ids = ids + [pad_id] * pad_len
    attention_mask = attention_mask + [0] * pad_len
    return ids, attention_mask


def compute_tiny_smiles_bert_embeddings(smiles_series, model_dir=None, batch_size=128):
    """Load the 5A Tiny SMILES-BERT model and compute CTRPv2 CLS embeddings."""
    if not TORCH_AVAILABLE:
        raise ImportError("Tiny SMILES-BERT embedding requires torch.")

    model_dir = Path(model_dir) if model_dir is not None else find_tiny_model_dir()
    with open(model_dir / "tiny_smiles_bert_vocab.json", "r", encoding="utf-8") as f:
        vocab_data = json.load(f)
    with open(model_dir / "tiny_smiles_bert_config.json", "r", encoding="utf-8") as f:
        config = json.load(f)

    stoi = vocab_data["stoi"]
    pad_id = stoi[config["pad_token"]]
    cls_id = stoi[config["cls_token"]]
    sep_id = stoi[config["sep_token"]]
    unk_id = stoi[config["unk_token"]]
    max_len = int(config["max_len"])

    tiny_model = TinySmilesBERT(
        vocab_size=int(config["vocab_size"]),
        max_len=max_len,
        pad_id=pad_id,
        dim=int(config["dim"]),
        n_heads=int(config["n_heads"]),
        n_layers=int(config["n_layers"]),
        dropout=float(config["dropout"]),
    ).to(device)
    state_dict = torch.load(model_dir / "tiny_smiles_bert_state_dict.pt", map_location=device)
    tiny_model.load_state_dict(state_dict)
    tiny_model.eval()

    encoded_ids = []
    encoded_masks = []
    unknown_counts = []
    for smiles in smiles_series.astype(str):
        tokens = tokenize_smiles_for_tiny_model(smiles)
        unknown_counts.append(sum(tok not in stoi for tok in tokens))
        ids, mask = encode_tokens_for_tiny_model(tokens, stoi, max_len, pad_id, cls_id, sep_id, unk_id)
        encoded_ids.append(ids)
        encoded_masks.append(mask)

    input_ids = torch.tensor(encoded_ids, dtype=torch.long)
    attention_masks = torch.tensor(encoded_masks, dtype=torch.bool)

    embeddings = []
    with torch.no_grad():
        for start in range(0, len(input_ids), batch_size):
            batch_ids = input_ids[start : start + batch_size].to(device)
            batch_masks = attention_masks[start : start + batch_size].to(device)
            _, hidden = tiny_model(batch_ids, batch_masks)
            cls_embedding = hidden[:, 0, :].cpu().numpy()
            embeddings.append(cls_embedding)

    print("Tiny model directory:", model_dir)
    print("Total unknown tokens:", int(np.sum(unknown_counts)))
    print("Compounds with at least one unknown token:", int(np.sum(np.array(unknown_counts) > 0)))
    return np.vstack(embeddings).astype(np.float32)


tiny_compounds = valid_fp_compounds
tiny_smiles = df_drug_info.loc[tiny_compounds, "cpd_smiles"]

X_tiny = compute_tiny_smiles_bert_embeddings(tiny_smiles, batch_size=128)
tiny_cols = [f"tinybert_{i}" for i in range(X_tiny.shape[1])]
df_tiny_features = pd.DataFrame(X_tiny, index=tiny_compounds, columns=tiny_cols)
print("Tiny SMILES-BERT embedding matrix:", df_tiny_features.shape)


### 2-7. AUC long-form table과 최종 feature table 만들기

In [ ]:
common_representation_compounds = (
    df_fp_features.index
    .intersection(df_chemberta_features.index)
    .intersection(df_tiny_features.index)
    .tolist()
)

auc_long = (
    df_auc.loc[usable_cell_lines, common_representation_compounds]
    .rename_axis("cell_line")
    .reset_index()
    .melt(id_vars="cell_line", var_name="compound", value_name="AUC")
    .dropna(subset=["AUC"])
    .reset_index(drop=True)
)
auc_long["AUC"] = auc_long["AUC"].astype(float)

model_df_base = auc_long.join(cellline_features, on="cell_line").dropna().reset_index(drop=True)
model_df_morgan = model_df_base.join(df_fp_features, on="compound").dropna().reset_index(drop=True)
model_df_chemberta = model_df_base.join(df_chemberta_features, on="compound").dropna().reset_index(drop=True)
model_df_tiny = model_df_base.join(df_tiny_features, on="compound").dropna().reset_index(drop=True)

morgan_feature_cols = cell_cols + fp_cols
chemberta_feature_cols = cell_cols + chemberta_cols
tiny_feature_cols = cell_cols + tiny_cols

print("Long-form AUC table:", auc_long.shape)
print("Morgan modeling table:", model_df_morgan.shape)
print("ChemBERTa modeling table:", model_df_chemberta.shape)
print("Tiny SMILES-BERT modeling table:", model_df_tiny.shape)
print("Morgan features:", len(morgan_feature_cols))
print("ChemBERTa features:", len(chemberta_feature_cols))
print("Tiny SMILES-BERT features:", len(tiny_feature_cols))


## 3. Train/validation/test split

두 ANN 모델이 같은 sample split을 사용하도록 `(cell_line, compound)` pair 기준으로 split id를 먼저 만듭니다. test set은 최종 평가에만 사용하고, train set 안에서 validation set을 다시 나눕니다.


In [ ]:
split_pairs = auc_long[["cell_line", "compound", "AUC"]].copy()
train_pairs, test_pairs = mod_sel.train_test_split(split_pairs, test_size=0.2, random_state=RANDOM_STATE, shuffle=True)
train_pairs, val_pairs = mod_sel.train_test_split(train_pairs, test_size=0.2, random_state=RANDOM_STATE, shuffle=True)

for name, df in [("train", train_pairs), ("validation", val_pairs), ("test", test_pairs)]:
    print(name, df.shape[0], "rows |", df["cell_line"].nunique(), "cell lines |", df["compound"].nunique(), "compounds")


In [ ]:
def subset_by_pairs(model_df, pair_df):
    """Select rows from a modeling table using a cell_line-compound pair table."""
    pair_index = pd.MultiIndex.from_frame(pair_df[["cell_line", "compound"]])
    model_index = pd.MultiIndex.from_frame(model_df[["cell_line", "compound"]])
    return model_df.loc[model_index.isin(pair_index)].copy()


splits = {
    "Morgan fingerprint": {
        "train": subset_by_pairs(model_df_morgan, train_pairs),
        "val": subset_by_pairs(model_df_morgan, val_pairs),
        "test": subset_by_pairs(model_df_morgan, test_pairs),
        "features": morgan_feature_cols,
    },
    "ChemBERTa embedding": {
        "train": subset_by_pairs(model_df_chemberta, train_pairs),
        "val": subset_by_pairs(model_df_chemberta, val_pairs),
        "test": subset_by_pairs(model_df_chemberta, test_pairs),
        "features": chemberta_feature_cols,
    },
    "Tiny SMILES-BERT embedding": {
        "train": subset_by_pairs(model_df_tiny, train_pairs),
        "val": subset_by_pairs(model_df_tiny, val_pairs),
        "test": subset_by_pairs(model_df_tiny, test_pairs),
        "features": tiny_feature_cols,
    },
}

for name, item in splits.items():
    print(name)
    print("  train/val/test:", item["train"].shape[0], item["val"].shape[0], item["test"].shape[0])
    print("  n_features:", len(item["features"]))


## 4. ANN 모델 정의

두 representation이 공정하게 비교되도록 같은 MLP 구조와 같은 early stopping 규칙을 사용합니다.


In [ ]:
if not TF_AVAILABLE:
    raise ImportError("ANN 섹션을 실행하려면 tensorflow가 필요합니다.")


def regression_scores(y_true, y_pred):
    """Calculate common regression metrics for observed and predicted AUC."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    if len(y_true) > 1 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        pcc = pearsonr(y_true, y_pred)[0]
        scc = spearmanr(y_true, y_pred)[0]
    else:
        pcc = np.nan
        scc = np.nan
    return {
        "PCC": pcc,
        "SCC": scc,
        "R2": met.r2_score(y_true, y_pred),
        "MAE": met.mean_absolute_error(y_true, y_pred),
        "RMSE": met.root_mean_squared_error(y_true, y_pred),
    }


def build_mlp_regressor(n_features, hidden_units=(256, 128), activation="relu", dropout_rate=0.2, learning_rate=0.001):
    """Build and compile a Keras MLP regressor for AUC prediction."""
    model = keras.Sequential(name="mlp_auc_regressor")
    model.add(layers.Input(shape=(n_features,), name="input_features"))
    for i, units in enumerate(hidden_units, start=1):
        model.add(layers.Dense(units, activation=activation, name=f"dense_{i}"))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate, name=f"dropout_{i}"))
    model.add(layers.Dense(1, activation="linear", name="auc_output"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


class CompactTrainingLogger(keras.callbacks.Callback):
    """Print compact train/validation metrics at the end of each epoch."""
    def __init__(self, total_epochs, label="model"):
        super().__init__()
        self.total_epochs = total_epochs
        self.label = label

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(
            f"{self.label} | Epoch {epoch + 1:03d}/{self.total_epochs} | "
            f"loss {logs.get('loss', float('nan')):.4f} | "
            f"mae {logs.get('mae', float('nan')):.4f} | "
            f"val_loss {logs.get('val_loss', float('nan')):.4f} | "
            f"val_mae {logs.get('val_mae', float('nan')):.4f}"
        )


## 5. ANN 학습과 평가


In [ ]:
def train_and_evaluate_ann(name, split_item, hidden_units=(256, 128), dropout_rate=0.2, max_epochs=80):
    """Train one ANN model and evaluate it on the shared test set."""
    feature_cols = split_item["features"]
    train_df = split_item["train"]
    val_df = split_item["val"]
    test_df = split_item["test"]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[feature_cols]).astype(np.float32)
    X_val = scaler.transform(val_df[feature_cols]).astype(np.float32)
    X_test = scaler.transform(test_df[feature_cols]).astype(np.float32)

    y_train = train_df["AUC"].astype(np.float32).to_numpy()
    y_val = val_df["AUC"].astype(np.float32).to_numpy()
    y_test = test_df["AUC"].astype(np.float32).to_numpy()

    model = build_mlp_regressor(
        n_features=X_train.shape[1],
        hidden_units=hidden_units,
        dropout_rate=dropout_rate,
        learning_rate=0.001,
    )
    early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=max_epochs,
        batch_size=128,
        callbacks=[early_stop, CompactTrainingLogger(max_epochs, label=name)],
        verbose=0,
    )

    y_pred = model.predict(X_test, verbose=0).ravel()
    scores = regression_scores(y_test, y_pred)
    result = {
        "representation": name,
        "n_features": len(feature_cols),
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_test": len(test_df),
        **scores,
    }
    pred_df = test_df[["cell_line", "compound", "AUC"]].copy()
    pred_df["predicted_AUC"] = y_pred
    return model, pd.DataFrame(history.history), result, pred_df


ann_outputs = {}
ann_results = []
ann_predictions = {}
for representation_name, split_item in splits.items():
    print("\nTraining ANN with", representation_name)
    model, history_df, result, pred_df = train_and_evaluate_ann(representation_name, split_item)
    ann_outputs[representation_name] = {"model": model, "history": history_df}
    ann_results.append(result)
    ann_predictions[representation_name] = pred_df

ann_result_df = pd.DataFrame(ann_results).sort_values("PCC", ascending=False)
ann_result_df

### 5-1. Learning curve 비교


In [ ]:
fig, axes = plt.subplots(1, len(ann_outputs), figsize=(5 * len(ann_outputs), 4), sharey=True)
axes = np.atleast_1d(axes)

for ax, (name, item) in zip(axes, ann_outputs.items()):
    history_df = item["history"]
    ax.plot(history_df["loss"], label="train loss")
    ax.plot(history_df["val_loss"], label="validation loss")
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE loss")
    ax.legend()
plt.tight_layout()


### 5-2. Test PCC 비교


In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=ann_result_df, x="representation", y="PCC", color="steelblue")
plt.ylabel("Test PCC (higher is better)")
plt.xlabel("")
plt.ylim(0, max(1.0, ann_result_df["PCC"].max() + 0.05))
plt.title("ANN performance by compound representation")
plt.tight_layout()

ann_result_df


### 5-3. True AUC와 predicted AUC 산점도


In [ ]:
fig, axes = plt.subplots(1, len(ann_predictions), figsize=(5 * len(ann_predictions), 4), sharex=True, sharey=True)
axes = np.atleast_1d(axes)

for ax, (name, pred_df) in zip(axes, ann_predictions.items()):
    y_true = pred_df["AUC"].to_numpy()
    y_pred = pred_df["predicted_AUC"].to_numpy()
    scores = regression_scores(y_true, y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    ax.scatter(y_true, y_pred, alpha=0.35, s=12)
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1)
    title_text = f"{name}\nPCC={scores['PCC']:.3f}, RMSE={scores['RMSE']:.3f}"
    ax.set_title(title_text)
    ax.set_xlabel("True AUC")
    ax.set_ylabel("Predicted AUC")
plt.tight_layout()


## 6. 해석과 토의

1. Morgan fingerprint, ChemBERTa embedding, Tiny SMILES-BERT embedding 중 어느 representation이 가장 높은 PCC를 보였나요?
2. 5A Tiny SMILES-BERT embedding의 성능이 낮다면 어떤 이유가 가능할까요?
3. QM9로 학습한 작은 모델을 CTRPv2 drug compound에 적용할 때 domain shift는 어떤 문제를 만들까요?
4. Tiny model에서 `[UNK]` token이 많아지면 embedding 품질은 어떻게 달라질까요?
5. Morgan fingerprint는 rule-based chemical prior를 가지고 있고, ChemBERTa는 pretrained learned representation입니다. 작은 downstream dataset에서는 어떤 표현이 더 안정적일까요?
6. ChemBERTa embedding과 Tiny SMILES-BERT embedding은 모두 dense vector입니다. feature 수와 parameter 수 관점에서 ANN 학습 난이도는 Morgan fingerprint와 어떻게 달라질까요?
